# Actividad de clases: obtener, transformar y resumir datos (proyecto TikTok)
**Integrante(s) / Grupo / Docente:** _completar_

Esta actividad convierte en **clases reutilizables** lo que se hizo con funciones en el notebook de `AE2/` y lo aplica al **mismo dataset y a la misma necesidad** del documento de AE2.

```
HFDatasetLoader → DataFrame (muestra) → transformación mínima → DatasetSummarizer → resultados/resumen_dataframe.txt
```
| Pieza | Archivo | Relación con AE2 |
|---|---|---|
| `HFDatasetLoader` | `hf_dataset_loader.py` | Reemplaza la lectura por *row groups* del notebook de AE2 (muestra de un dataset de 460 GB) |
| `DatasetSummarizer` | `dataset_summarizer.py` | Reemplaza la función `generar_resumen` (filas, columnas, tipos, nulos y vacíos) |
| Transformación | este notebook | Conserva las variables del diseño inicial del análisis (Documento AE2, sección 6.6) |

> Los textos libres (`caption`, `mentions`, `on_screen_text`) e ids de creadores no se cargan, por privacidad (ver AE2, sección 6.2).

## 1. Importar las clases

In [ ]:
import pandas as pd

from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

## 2. Obtener el DataFrame desde Hugging Face
Mismo dataset y parámetros que AE2: `datasocial/tiktok-5.6B-videos`, split `train`, muestra de hasta 200.000 filas (semilla 42). El dataset completo tiene 5.598.141.717 filas (según la ficha), así que **se trabaja con una muestra**.

In [ ]:
DATASET_ID = "datasocial/tiktok-5.6B-videos"
SPLIT = "train"
TOTAL_FILAS_DATASET = 5_598_141_717   # según la ficha
N_FILAS, SEED = 200_000, 42

COLUMNAS_CARGADAS = ["video_id","posted_at","country","language","type","duration_ms","image_count",
    "width","height","hashtags","sound_id","is_ad","branded_content","product_id","is_shop_video",
    "seller_id","is_ai_generated","not_recommended_for_fyp","sound_muted","views","likes","comments",
    "shares","saves","downloads","engagement_rate","stats_updated_at"]

loader = HFDatasetLoader(DATASET_ID, split=SPLIT, columns=COLUMNAS_CARGADAS, max_rows=N_FILAS, seed=SEED)
df = loader.load()

print(loader)
print("Revisión:", loader.revision)
print("Muestra:", loader.sample_note)
print("DataFrame original:", df.shape)

In [ ]:
df.head()

## 3. Transformación mínima
**Selección de columnas:** se conservan solo las 9 variables del diseño inicial del análisis de AE2 (Tabla 3 del documento), necesarias para la pregunta *¿cómo varían las vistas medianas y la tasa de interacción según tipo, duración, país y carácter comercial?*  La operación devuelve un DataFrame nuevo, `df_transformado`, que es el que se resume.

In [ ]:
VARIABLES_ANALISIS = ["posted_at", "type", "duration_ms", "country", "is_ad", "is_shop_video",
                      "branded_content", "views", "engagement_rate"]

df_transformado = df[VARIABLES_ANALISIS]

print(f"Columnas antes: {df.shape[1]} | después: {df_transformado.shape[1]}")
print(f"Filas: {df_transformado.shape[0]} (sin cambio)")
df_transformado.head()

## 4. Generar el resumen (.txt) del DataFrame **transformado**

In [ ]:
summarizer = DatasetSummarizer(output_dir="resultados", filename="resumen_dataframe.txt")

ruta = summarizer.summarize(
    df_transformado,
    source=f"{DATASET_ID} (split: {loader.split_used})",
    description=f"selección de {len(VARIABLES_ANALISIS)} columnas del diseño de análisis: {', '.join(VARIABLES_ANALISIS)}",
    metadata={
        "Enlace": f"https://huggingface.co/datasets/{DATASET_ID}",
        "Licencia (según la ficha)": "CC BY-NC 4.0",
        "Revisión (commit)": loader.revision,
        "Fecha de consulta": pd.Timestamp.today().date().isoformat(),
        "Filas del dataset completo (ficha)": f"{TOTAL_FILAS_DATASET:,}",
        "¿Muestra?": "Sí. " + loader.sample_note if loader.is_sample else "No",
        "Periodo de la muestra (posted_at)": f"{df_transformado['posted_at'].min()} a {df_transformado['posted_at'].max()}",
    },
)
print("Archivo generado en:", ruta.resolve())

## 5. Resultado y verificación

In [ ]:
texto = ruta.read_text(encoding="utf-8")
print(texto)

In [ ]:
# El resumen debe corresponder al DataFrame transformado (no al original)
assert f"- Filas: {len(df_transformado)}" in texto
assert f"- Columnas: {df_transformado.shape[1]}" in texto
assert "caption" not in texto and "likes" not in texto   # columnas que ya no están
print("OK: el resumen coincide con df_transformado")

## 6. Qué transformación se hizo y cómo se relaciona con AE2
Se hizo una **selección de columnas**: de las 27 cargadas se pasó a las 9 que usa la pregunta de análisis (`views` y `engagement_rate` como resultado; `type`, `duration_ms`, `country`, `is_ad`, `is_shop_video`, `branded_content` como variables de comparación; `posted_at` para conocer el periodo). El resumen `.txt` describe esos datos **ya reducidos**: mismas filas, menos columnas.

Los valores nulos y vacíos que aparecen en el resumen (p. ej. `country` o `branded_content` en blanco, nulos en `engagement_rate`) alimentan la decisión pendiente del paso 2 del diseño de AE2: cómo tratar nulos, ceros con significado y registros antiguos.

*Las clases son genéricas:* con otro dataset basta cambiar `DATASET_ID` (por ejemplo `HFDatasetLoader("scikit-learn/iris", split="train")` carga el dataset completo, sin `max_rows`).